In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSV files into memory (kept in kernel variables)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Timezone: convert timestamp -> timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# --- Metric summary ---
total_rows_metric = df_metric.shape[0]
metric_cmdb_ids = pd.Series(sorted(df_metric['cmdb_id'].unique()))[:20].reset_index(drop=True)
metric_kpi_names = pd.Series(sorted(df_metric['kpi_name'].unique()))[:50].reset_index(drop=True)

# Group and compute stats (global thresholds) for metric
mg = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_agg = mg.agg(count='count', min='min', max='max', mean='mean', median='median').reset_index()
metric_p75 = mg.quantile(0.75).reset_index(name='p75')
metric_p95 = mg.quantile(0.95).reset_index(name='p95')
metric_summary = metric_agg.merge(metric_p75, on=['cmdb_id','kpi_name']).merge(metric_p95, on=['cmdb_id','kpi_name'])

# format and pick top 20 by count
metric_top20 = metric_summary.sort_values('count', ascending=False).head(20).copy()
num_cols = ['min','max','mean','median','p75','p95']
metric_top20[num_cols] = metric_top20[num_cols].applymap(lambda x: float(np.round(x, 6)) if pd.notnull(x) else x)

# --- Trace summary ---
total_rows_trace = df_trace.shape[0]
trace_cmdb_ids = pd.Series(sorted(df_trace['cmdb_id'].unique()))[:20].reset_index(drop=True)
trace_names = pd.Series(sorted(df_trace['trace_name'].unique())).reset_index(drop=True)

tg = df_trace.groupby(['cmdb_id','trace_name'])['value']
trace_agg = tg.agg(count='count', min='min', max='max', mean='mean', median='median').reset_index()
trace_p75 = tg.quantile(0.75).reset_index(name='p75')
trace_p95 = tg.quantile(0.95).reset_index(name='p95')
trace_summary = trace_agg.merge(trace_p75, on=['cmdb_id','trace_name']).merge(trace_p95, on=['cmdb_id','trace_name'])
trace_top20 = trace_summary.sort_values('count', ascending=False).head(20).copy()
trace_top20[num_cols] = trace_top20[num_cols].applymap(lambda x: float(np.round(x, 6)) if pd.notnull(x) else x)

# --- Log summary ---
total_rows_log = df_log.shape[0]
log_cmdb_ids = pd.Series(sorted(df_log['cmdb_id'].unique()))[:20].reset_index(drop=True)
log_names = pd.Series(sorted(df_log['log_name'].unique())).reset_index(drop=True)

log_top20 = pd.DataFrame()
if 'value' in df_log.columns:
    lg = df_log.groupby(['cmdb_id','log_name'])['value']
    log_agg = lg.agg(count='count', min='min', max='max', mean='mean', median='median').reset_index()
    log_p75 = lg.quantile(0.75).reset_index(name='p75')
    log_p95 = lg.quantile(0.95).reset_index(name='p95')
    log_summary = log_agg.merge(log_p75, on=['cmdb_id','log_name']).merge(log_p95, on=['cmdb_id','log_name'])
    log_top20 = log_summary.sort_values('count', ascending=False).head(20).copy()
    log_top20[num_cols] = log_top20[num_cols].applymap(lambda x: float(np.round(x, 6)) if pd.notnull(x) else x)

# --- Error logs summary ---
total_rows_error = df_error.shape[0]
error_cmdb_ids = pd.Series(sorted(df_error['cmdb_id'].unique())).reset_index(drop=True)
# sample up to 5 messages with timestamps and cmdb_id (use dt formatted)
if 'message' in df_error.columns:
    error_samples = df_error.sort_values('timestamp').head(5).loc[:, ['dt','cmdb_id','message']].copy()
    # format dt to readable string in UTC+8
    error_samples['dt'] = error_samples['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_samples = pd.DataFrame(columns=['dt','cmdb_id','message'])

# --- Build compact textual summary ---
def tab_to_text(df, cols, max_rows=20):
    # convert a small dataframe to a compact text table
    if df.empty:
        return "(no rows)"
    txt = df[cols].head(max_rows).to_string(index=False)
    return txt

summary_lines = []
summary_lines.append(f"metric.csv: total_rows={total_rows_metric}")
summary_lines.append(f"metric.csv: up to 20 cmdb_id -> {', '.join(metric_cmdb_ids.astype(str).tolist())}")
summary_lines.append(f"metric.csv: up to 50 kpi_name -> {', '.join(metric_kpi_names.astype(str).tolist())}")
summary_lines.append("metric.csv: top 20 (cmdb_id, kpi_name) by count with stats (count,min,max,mean,median,p75,p95):")
summary_lines.append(tab_to_text(metric_top20, ['cmdb_id','kpi_name','count','min','max','mean','median','p75','p95'], max_rows=20))

summary_lines.append("")  # blank line
summary_lines.append(f"trace.csv: total_rows={total_rows_trace}")
summary_lines.append(f"trace.csv: up to 20 cmdb_id -> {', '.join(trace_cmdb_ids.astype(str).tolist())}")
summary_lines.append(f"trace.csv: trace_name values -> {', '.join(trace_names.astype(str).tolist())}")
summary_lines.append("trace.csv: top 20 (cmdb_id, trace_name) by count with stats (count,min,max,mean,median,p75,p95):")
summary_lines.append(tab_to_text(trace_top20, ['cmdb_id','trace_name','count','min','max','mean','median','p75','p95'], max_rows=20))

summary_lines.append("")
summary_lines.append(f"log.csv: total_rows={total_rows_log}")
summary_lines.append(f"log.csv: up to 20 cmdb_id -> {', '.join(log_cmdb_ids.astype(str).tolist())}")
summary_lines.append(f"log.csv: log_name values -> {', '.join(log_names.astype(str).tolist())}")
if not log_top20.empty:
    summary_lines.append("log.csv: top 20 (cmdb_id, log_name) by count with stats (count,min,max,mean,median,p75,p95):")
    summary_lines.append(tab_to_text(log_top20, ['cmdb_id','log_name','count','min','max','mean','median','p75','p95'], max_rows=20))
else:
    summary_lines.append("log.csv: no numeric 'value' column found or no rows to summarize.")

summary_lines.append("")
summary_lines.append(f"error_logs.csv: total_rows={total_rows_error}")
summary_lines.append(f"error_logs.csv: unique cmdb_id values -> {', '.join(error_cmdb_ids.astype(str).tolist())}")
summary_lines.append("error_logs.csv: up to 5 sample messages (dt in UTC+8, cmdb_id, message):")
if not error_samples.empty:
    # build compact lines
    sample_lines = []
    for _, r in error_samples.iterrows():
        msg = r['message']
        # shorten message if too long
        if isinstance(msg, str) and len(msg) > 160:
            msg = msg[:157] + '...'
        sample_lines.append(f"{r['dt']} | {r['cmdb_id']} | {msg}")
    summary_lines.extend(sample_lines)
else:
    summary_lines.append("(no error messages)")

summary_text = "\n".join(summary_lines)

# Display the compact textual summary as the final output; keep all full dataframes and summary tables as variables for later use.
summary_text

```
Out[1]:
```
```python
# Summarize the prior execution results in plain English (display summary_text)
summary_text = (
    "Summary of loaded telemetry files:\n\n"
    "1) metric.csv:\n"
    "- Total rows: 321,411.\n"
    "- Contains many cmdb_id values (examples shown up to 20, including adservice, cartservice, checkoutservice, shippingservice2-0, etc.).\n"
    "- Many distinct KPI names (examples shown up to 50, including app.grpc.*, app.http.*, and many container.* metrics).\n"
    "- Top (cmdb_id, kpi_name) pairs by row count are dominated by shippingservice2-0 entries (each with count=61 for the top entries shown). Global statistics (min, max, mean, median, 75th, 95th percentiles) were computed per (cmdb_id, kpi_name) and the top 20 pairs by count were returned.\n\n"
    "2) trace.csv:\n"
    "- Total rows: 30,208.\n"
    "- Up to 20 cmdb_id examples include many service pods (e.g., adservice-0/1/2, shippingservice-1, recommendationservice-0, etc.).\n"
    "- Many trace_name values (listed in the output), covering trace.from_*, trace.to_*, trace.self.*, with metrics like duration_mean, duration_p95, error_rate, row_count.\n"
    "- Top (cmdb_id, trace_name) pairs by count show many entries with count=30 (examples include trace.from_frontend*.duration_mean and row_count for various services). Per-pair global statistics were computed and the top 20 pairs by count were returned.\n\n"
    "3) log.csv:\n"
    "- Total rows: 1,774.\n"
    "- Up to 20 cmdb_id examples include frontend-*, cartservice-*, currencyservice-*, etc.\n"
    "- log_name values present: log.row_count and log.error_count.\n"
    "- Top pairs by count (up to 20 shown) typically have count=31 for many pods (e.g., cartservice-2 log.row_count = 31). log.row_count shows nonzero counts and ranges; log.error_count is zero in the shown top pairs.\n\n"
    "4) error_logs.csv:\n"
    "- Total rows: 0 (no error log records present). No sample messages available.\n\n"
    "Notes:\n"
    "- All per-(cmdb_id, KPI/trace/log) global thresholds (median, 75th, 95th percentiles) were computed across the entire series before any time filtering.\n"
    "- Full DataFrames remain loaded in kernel variables (df_metric, df_trace, df_log, df_error) for further analysis.\n"
    "If you want, I can now: (a) inspect specific (cmdb_id, kpi_name) or trace/log pairs in detail, (b) compute anomaly thresholds for particular series, or (c) filter the data by time range and produce focused summaries."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

metric.csv: total_rows=321411
metric.csv: up to 20 cmdb_id -> adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice, currencyservice-0, currencyservice-1, currencyservice-2
metric.csv: up to 50 kpi_name -> app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, app.http.mrt, app.http.rr, app.http.sr, container.node-5.container_cpu_cfs_periods, container.node-5.container_cpu_cfs_throttled_periods, container.node-5.container_cpu_cfs_throttled_seconds, container.node-5.container_cpu_load_average_10s, container.node-5.container_cpu_system_seconds, container.node-5.container_cpu_usage_seconds, container.node-5.container_cpu_user_seconds, container.node-5.container_file_descriptors, container.node-5.container_fs_inodes./dev/vda1, container.node-5.container_fs_inodes_free./dev/vda1, container.node-5.container_fs_io_current./dev/vda1, container.node-5.container_fs_io_time_seconds./dev/vda1, container.node-5.container_fs_io_time_weighted_seconds./dev/vda1, container.node-5.container_fs_limit_MB./dev/vda1, container.node-5.container_fs_read_seconds./dev/vda1, container.node-5.container_fs_reads./dev/vda, container.node-5.container_fs_reads./dev/vda1, container.node-5.container_fs_reads_MB./dev/vda, container.node-5.container_fs_reads_merged./dev/vda1, container.node-5.container_fs_sector_reads./dev/vda1, container.node-5.container_fs_sector_writes./dev/vda1, container.node-5.container_fs_usage_MB./dev/vda1, container.node-5.container_fs_write_seconds./dev/vda1, container.node-5.container_fs_writes./dev/vda, container.node-5.container_fs_writes./dev/vda1, container.node-5.container_fs_writes_MB./dev/vda, container.node-5.container_fs_writes_merged./dev/vda1, container.node-5.container_last_seen, container.node-5.container_memory_cache, container.node-5.container_memory_failcnt, container.node-5.container_memory_failures.container.pgfault, container.node-5.container_memory_failures.container.pgmajfault, container.node-5.container_memory_failures.hierarchy.pgfault, container.node-5.container_memory_failures.hierarchy.pgmajfault, container.node-5.container_memory_mapped_file, container.node-5.container_memory_max_usage_MB, container.node-5.container_memory_rss, container.node-5.container_memory_swap, container.node-5.container_memory_usage_MB, container.node-5.container_memory_working_set_MB, container.node-5.container_network_receive_MB.eth0, container.node-5.container_network_receive_errors.eth0
metric.csv: top 20 (cmdb_id, kpi_name) by count with stats (count,min,max,mean,median,p75,p95):
           cmdb_id                                                                     kpi_name  count          min          max         mean       median          p75          p95
shippingservice2-0 mesh.source.shippingservice2.jaeger-collector.istio_response_bytes.http.202.     61 1.990000e+02 1.293500e+03 7.087336e+02 6.965000e+02 8.457500e+02 1.144250e+03
shippingservice2-0                           container.node-6.container_network_receive_MB.eth0     61 2.183300e-02 3.361630e-01 4.836600e-02 2.894700e-02 3.196300e-02 3.151370e-01
shippingservice2-0                       container.node-6.container_network_receive_errors.eth0     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                      container.node-6.container_network_receive_packets.eth0     61 2.760000e+02 4.390000e+02 3.273033e+02 3.280000e+02 3.485000e+02 3.870000e+02
shippingservice2-0              container.node-6.container_network_receive_packets_dropped.eth0     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                          container.node-6.container_network_transmit_MB.eth0     61 3.380230e-01 5.529370e-01 4.475520e-01 4.466590e-01 4.545140e-01 4.807920e-01
shippingservice2-0                      container.node-6.container_network_transmit_errors.eth0     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                     container.node-6.container_network_transmit_packets.eth0     61 1.575000e+02 3.035000e+02 2.053525e+02 2.025000e+02 2.165000e+02 2.530000e+02
shippingservice2-0             container.node-6.container_network_transmit_packets_dropped.eth0     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                 container.node-6.container_memory_failures.hierarchy.pgfault     61 0.000000e+00 3.300000e+01 5.409840e-01 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0              container.node-6.container_memory_failures.hierarchy.pgmajfault     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                                container.node-6.container_memory_mapped_file     61 0.000000e+00 2.703360e+05 4.431738e+04 0.000000e+00 0.000000e+00 2.703360e+05
shippingservice2-0                               container.node-6.container_memory_max_usage_MB     61 1.287266e+02 1.287266e+02 1.287266e+02 1.287266e+02 1.287266e+02 1.287266e+02
shippingservice2-0                                        container.node-6.container_memory_rss     61 1.000243e+07 1.000243e+07 1.000243e+07 1.000243e+07 1.000243e+07 1.000243e+07
shippingservice2-0                                       container.node-6.container_memory_swap     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                                   container.node-6.container_memory_usage_MB     61 3.137500e+01 3.233984e+01 3.164075e+01 3.157422e+01 3.157422e+01 3.233984e+01
shippingservice2-0                             container.node-6.container_memory_working_set_MB     61 1.909375e+01 1.961719e+01 1.935300e+01 1.942383e+01 1.961719e+01 1.961719e+01
shippingservice2-0                               container.node-6.container_fs_writes./dev/vda1     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                             container.node-6.container_fs_writes_MB./dev/vda     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
shippingservice2-0                        container.node-6.container_fs_writes_merged./dev/vda1     61 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00

trace.csv: total_rows=30208
trace.csv: up to 20 cmdb_id -> adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice-0, emailservice-1, emailservice-2, emailservice2-0
trace.csv: trace_name values -> trace.from_checkoutservice-0.duration_mean, trace.from_checkoutservice-0.duration_p95, trace.from_checkoutservice-0.error_rate, trace.from_checkoutservice-0.row_count, trace.from_checkoutservice-1.duration_mean, trace.from_checkoutservice-1.duration_p95, trace.from_checkoutservice-1.error_rate, trace.from_checkoutservice-1.row_count, trace.from_checkoutservice-2.duration_mean, trace.from_checkoutservice-2.duration_p95, trace.from_checkoutservice-2.error_rate, trace.from_checkoutservice-2.row_count, trace.from_checkoutservice2-0.duration_mean, trace.from_checkoutservice2-0.duration_p95, trace.from_checkoutservice2-0.error_rate, trace.from_checkoutservice2-0.row_count, trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count, trace.from_frontend-1.duration_mean, trace.from_frontend-1.duration_p95, trace.from_frontend-1.error_rate, trace.from_frontend-1.row_count, trace.from_frontend-2.duration_mean, trace.from_frontend-2.duration_p95, trace.from_frontend-2.error_rate, trace.from_frontend-2.row_count, trace.from_frontend2-0.duration_mean, trace.from_frontend2-0.duration_p95, trace.from_frontend2-0.error_rate, trace.from_frontend2-0.row_count, trace.from_recommendationservice-0.duration_mean, trace.from_recommendationservice-0.duration_p95, trace.from_recommendationservice-0.error_rate, trace.from_recommendationservice-0.row_count, trace.from_recommendationservice-1.duration_mean, trace.from_recommendationservice-1.duration_p95, trace.from_recommendationservice-1.error_rate, trace.from_recommendationservice-1.row_count, trace.from_recommendationservice-2.duration_mean, trace.from_recommendationservice-2.duration_p95, trace.from_recommendationservice-2.error_rate, trace.from_recommendationservice-2.row_count, trace.from_recommendationservice2-0.duration_mean, trace.from_recommendationservice2-0.duration_p95, trace.from_recommendationservice2-0.error_rate, trace.from_recommendationservice2-0.row_count, trace.from_root.duration_mean, trace.from_root.duration_p95, trace.from_root.error_rate, trace.from_root.row_count, trace.self.duration_mean, trace.self.duration_p95, trace.self.error_rate, trace.self.row_count, trace.to_adservice-0.duration_mean, trace.to_adservice-0.duration_p95, trace.to_adservice-0.error_rate, trace.to_adservice-0.row_count, trace.to_adservice-1.duration_mean, trace.to_adservice-1.duration_p95, trace.to_adservice-1.error_rate, trace.to_adservice-1.row_count, trace.to_adservice-2.duration_mean, trace.to_adservice-2.duration_p95, trace.to_adservice-2.error_rate, trace.to_adservice-2.row_count, trace.to_adservice2-0.duration_mean, trace.to_adservice2-0.duration_p95, trace.to_adservice2-0.error_rate, trace.to_adservice2-0.row_count, trace.to_cartservice-0.duration_mean, trace.to_cartservice-0.duration_p95, trace.to_cartservice-0.error_rate, trace.to_cartservice-0.row_count, trace.to_cartservice-1.duration_mean, trace.to_cartservice-1.duration_p95, trace.to_cartservice-1.error_rate, trace.to_cartservice-1.row_count, trace.to_cartservice-2.duration_mean, trace.to_cartservice-2.duration_p95, trace.to_cartservice-2.error_rate, trace.to_cartservice-2.row_count, trace.to_cartservice2-0.duration_mean, trace.to_cartservice2-0.duration_p95, trace.to_cartservice2-0.error_rate, trace.to_cartservice2-0.row_count, trace.to_checkoutservice-0.duration_mean, trace.to_checkoutservice-0.duration_p95, trace.to_checkoutservice-0.error_rate, trace.to_checkoutservice-0.row_count, trace.to_checkoutservice-1.duration_mean, trace.to_checkoutservice-1.duration_p95, trace.to_checkoutservice-1.error_rate, trace.to_checkoutservice-1.row_count, trace.to_checkoutservice-2.duration_mean, trace.to_checkoutservice-2.duration_p95, trace.to_checkoutservice-2.error_rate, trace.to_checkoutservice-2.row_count, trace.to_checkoutservice2-0.duration_mean, trace.to_checkoutservice2-0.duration_p95, trace.to_checkoutservice2-0.error_rate, trace.to_checkoutservice2-0.row_count, trace.to_currencyservice-0.duration_mean, trace.to_currencyservice-0.duration_p95, trace.to_currencyservice-0.error_rate, trace.to_currencyservice-0.row_count, trace.to_currencyservice-1.duration_mean, trace.to_currencyservice-1.duration_p95, trace.to_currencyservice-1.error_rate, trace.to_currencyservice-1.row_count, trace.to_currencyservice-2.duration_mean, trace.to_currencyservice-2.duration_p95, trace.to_currencyservice-2.error_rate, trace.to_currencyservice-2.row_count, trace.to_currencyservice2-0.duration_mean, trace.to_currencyservice2-0.duration_p95, trace.to_currencyservice2-0.error_rate, trace.to_currencyservice2-0.row_count, trace.to_emailservice-0.duration_mean, trace.to_emailservice-0.duration_p95, trace.to_emailservice-0.error_rate, trace.to_emailservice-0.row_count, trace.to_emailservice-1.duration_mean, trace.to_emailservice-1.duration_p95, trace.to_emailservice-1.error_rate, trace.to_emailservice-1.row_count, trace.to_emailservice-2.duration_mean, trace.to_emailservice-2.duration_p95, trace.to_emailservice-2.error_rate, trace.to_emailservice-2.row_count, trace.to_emailservice2-0.duration_mean, trace.to_emailservice2-0.duration_p95, trace.to_emailservice2-0.error_rate, trace.to_emailservice2-0.row_count, trace.to_paymentservice-0.duration_mean, trace.to_paymentservice-0.duration_p95, trace.to_paymentservice-0.error_rate, trace.to_paymentservice-0.row_count, trace.to_paymentservice-1.duration_mean, trace.to_paymentservice-1.duration_p95, trace.to_paymentservice-1.error_rate, trace.to_paymentservice-1.row_count, trace.to_paymentservice-2.duration_mean, trace.to_paymentservice-2.duration_p95, trace.to_paymentservice-2.error_rate, trace.to_paymentservice-2.row_count, trace.to_paymentservice2-0.duration_mean, trace.to_paymentservice2-0.duration_p95, trace.to_paymentservice2-0.error_rate, trace.to_paymentservice2-0.row_count, trace.to_productcatalogservice-0.duration_mean, trace.to_productcatalogservice-0.duration_p95, trace.to_productcatalogservice-0.error_rate, trace.to_productcatalogservice-0.row_count, trace.to_productcatalogservice-1.duration_mean, trace.to_productcatalogservice-1.duration_p95, trace.to_productcatalogservice-1.error_rate, trace.to_productcatalogservice-1.row_count, trace.to_productcatalogservice-2.duration_mean, trace.to_productcatalogservice-2.duration_p95, trace.to_productcatalogservice-2.error_rate, trace.to_productcatalogservice-2.row_count, trace.to_productcatalogservice2-0.duration_mean, trace.to_productcatalogservice2-0.duration_p95, trace.to_productcatalogservice2-0.error_rate, trace.to_productcatalogservice2-0.row_count, trace.to_recommendationservice-0.duration_mean, trace.to_recommendationservice-0.duration_p95, trace.to_recommendationservice-0.error_rate, trace.to_recommendationservice-0.row_count, trace.to_recommendationservice-1.duration_mean, trace.to_recommendationservice-1.duration_p95, trace.to_recommendationservice-1.error_rate, trace.to_recommendationservice-1.row_count, trace.to_recommendationservice-2.duration_mean, trace.to_recommendationservice-2.duration_p95, trace.to_recommendationservice-2.error_rate, trace.to_recommendationservice-2.row_count, trace.to_recommendationservice2-0.duration_mean, trace.to_recommendationservice2-0.duration_p95, trace.to_recommendationservice2-0.error_rate, trace.to_recommendationservice2-0.row_count, trace.to_shippingservice-0.duration_mean, trace.to_shippingservice-0.duration_p95, trace.to_shippingservice-0.error_rate, trace.to_shippingservice-0.row_count, trace.to_shippingservice-1.duration_mean, trace.to_shippingservice-1.duration_p95, trace.to_shippingservice-1.error_rate, trace.to_shippingservice-1.row_count, trace.to_shippingservice-2.duration_mean, trace.to_shippingservice-2.duration_p95, trace.to_shippingservice-2.error_rate, trace.to_shippingservice-2.row_count, trace.to_shippingservice2-0.duration_mean, trace.to_shippingservice2-0.duration_p95, trace.to_shippingservice2-0.error_rate, trace.to_shippingservice2-0.row_count
trace.csv: top 20 (cmdb_id, trace_name) by count with stats (count,min,max,mean,median,p75,p95):
           cmdb_id                          trace_name  count      min       max     mean   median      p75       p95
shippingservice2-0    trace.from_frontend2-0.row_count     30 3.000000  8.000000 5.166667 5.000000 6.000000  6.550000
       adservice-0 trace.from_frontend-0.duration_mean     30 0.000016  0.000025 0.000019 0.000018 0.000019  0.000023
       adservice-0  trace.from_frontend-0.duration_p95     30 0.000018  0.000050 0.000023 0.000022 0.000023  0.000037
       adservice-0    trace.from_frontend-0.error_rate     30 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
       adservice-0     trace.from_frontend-0.row_count     30 2.000000  7.000000 5.166667 5.000000 6.000000  7.000000
       adservice-0 trace.from_frontend-1.duration_mean     30 0.000016  0.000049 0.000021 0.000019 0.000020  0.000032
       adservice-0  trace.from_frontend-1.duration_p95     30 0.000018  0.000189 0.000032 0.000022 0.000025  0.000085
       adservice-0    trace.from_frontend-1.error_rate     30 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
       adservice-0     trace.from_frontend-1.row_count     30 2.000000 11.000000 7.133333 7.500000 8.000000 10.000000
       adservice-0 trace.from_frontend-2.duration_mean     30 0.000016  0.000251 0.000026 0.000018 0.000019  0.000022
       adservice-0  trace.from_frontend-2.duration_p95     30 0.000017  0.001218 0.000062 0.000021 0.000024  0.000037
       adservice-0    trace.from_frontend-2.error_rate     30 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
       adservice-0     trace.from_frontend-2.row_count     30 5.000000 10.000000 7.800000 8.000000 9.000000 10.000000
       adservice-1 trace.from_frontend-0.duration_mean     30 0.000015  0.000036 0.000018 0.000017 0.000018  0.000023
       adservice-1  trace.from_frontend-0.duration_p95     30 0.000016  0.000108 0.000023 0.000018 0.000021  0.000045
       adservice-1    trace.from_frontend-0.error_rate     30 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
 shippingservice-1 trace.from_frontend-2.duration_mean     30 0.000066  0.000161 0.000078 0.000072 0.000077  0.000118
 shippingservice-1  trace.from_frontend-2.duration_p95     30 0.000066  0.000316 0.000092 0.000075 0.000082  0.000202
 shippingservice-1    trace.from_frontend-2.error_rate     30 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
 shippingservice-1     trace.from_frontend-2.row_count     30 1.000000  4.000000 2.633333 3.000000 3.000000  4.000000

log.csv: total_rows=1774
log.csv: up to 20 cmdb_id -> adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, cartservice-2, checkoutservice-0, checkoutservice-1, checkoutservice-2, currencyservice-0, currencyservice-1, currencyservice-2, emailservice-0, emailservice-1, emailservice-2, frontend-0, frontend-1, frontend-2, paymentservice-0, paymentservice-1
log.csv: log_name values -> log.error_count, log.row_count
log.csv: top 20 (cmdb_id, log_name) by count with stats (count,min,max,mean,median,p75,p95):
                cmdb_id        log_name  count  min   max       mean  median   p75   p95
          cartservice-2   log.row_count     31 14.0 396.0 313.129032   328.0 345.5 368.0
          cartservice-2 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
          cartservice-1   log.row_count     31  9.0 398.0 313.516129   330.0 345.5 374.0
          cartservice-1 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
          cartservice-0   log.row_count     31  9.0 381.0 312.612903   325.0 345.0 374.5
          cartservice-0 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
      currencyservice-0 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
      currencyservice-0   log.row_count     31  4.0 346.0 271.580645   285.0 308.0 324.5
      currencyservice-2 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
      currencyservice-2   log.row_count     31  3.0 357.0 271.354839   278.0 308.5 329.5
      currencyservice-1 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
      currencyservice-1   log.row_count     31  1.0 353.0 271.612903   282.0 310.5 332.0
             frontend-2 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
             frontend-2   log.row_count     31  9.0 796.0 587.032258   593.0 632.0 685.5
recommendationservice-0   log.row_count     31  1.0 103.0  81.516129    84.0  89.5  94.5
recommendationservice-0 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
recommendationservice-2 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
recommendationservice-2   log.row_count     31  1.0  95.0  81.548387    87.0  89.0  92.5
      shippingservice-1 log.error_count     31  0.0   0.0   0.000000     0.0   0.0   0.0
      shippingservice-1   log.row_count     31  2.0  59.0  35.806452    37.0  42.5  55.5

error_logs.csv: total_rows=0
error_logs.csv: unique cmdb_id values -> 
error_logs.csv: up to 5 sample messages (dt in UTC+8, cmdb_id, message):
(no error messages)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric loaded earlier in the kernel.
# Ensure dt exists and is timezone-aware in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure timezone is Asia/Shanghai for dt column
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = df_metric['dt'].dt.tz_localize(pytz.UTC).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# 1) Compute global p95 and p5 per (cmdb_id, kpi_name) using the full series
thr_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
thr_p5  = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).reset_index(name='p5')
thresholds_df = thr_p95.merge(thr_p5, on=['cmdb_id','kpi_name'])

# 2) Filter df_metric to incident window March 20, 2022 22:00 to 22:30 (UTC+8)
start = pd.Timestamp('2022-03-20 22:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 22:30:00', tz=tz)
df_window = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# 3) Merge thresholds into windowed rows to evaluate anomalies (thresholds computed globally above)
df_w_thr = df_window.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left')

# If some (cmdb_id,kpi_name) in-window have no global thresholds (unlikely), drop them
df_w_thr = df_w_thr.dropna(subset=['p95','p5'])

# 4) Flag anomalies per row
df_w_thr['is_high'] = df_w_thr['value'] >= df_w_thr['p95']
df_w_thr['is_low']  = df_w_thr['value'] <= df_w_thr['p5']
df_w_thr['anom_flag'] = df_w_thr['is_high'] | df_w_thr['is_low']

# 5) Aggregate per (cmdb_id, kpi_name)
grp = df_w_thr.groupby(['cmdb_id','kpi_name'])
agg = grp.agg(
    count_high = ('is_high','sum'),
    count_low  = ('is_low','sum'),
    count_in_window = ('value','size'),
    max_in_window = ('value','max'),
    min_in_window = ('value','min'),
    earliest_anom_ts = ('dt', lambda s: s[df_w_thr.loc[s.index,'anom_flag']].min() if len(s)>0 else pd.NaT),
    p95 = ('p95','first'),
    p5  = ('p5','first'),
).reset_index()

# Note: The lambda above uses indices from group series; ensure correctness by alternative safer computation:
# recompute earliest_anom_ts using a separate grouped operation to be robust
anom_rows = df_w_thr[df_w_thr['anom_flag']]
if not anom_rows.empty:
    earliest_per_pair = anom_rows.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index(name='earliest_anom_ts')
    agg = agg.drop(columns=['earliest_anom_ts']).merge(earliest_per_pair, on=['cmdb_id','kpi_name'], how='left')
else:
    # no anomalies => set earliest_anom_ts column to NaT
    agg['earliest_anom_ts'] = pd.NaT

# 6) Keep only pairs with any anomaly in-window
agg['anom_count'] = agg['count_high'] + agg['count_low']
anomalies_df = agg[agg['anom_count'] > 0].copy()

# 7) Compute severity metrics (handle divide-by-zero carefully)
# severity_high = max_in_window / p95  (if count_high>0)
# severity_low  = p5 / min_in_window    (if count_low>0)
def compute_sev_high(row):
    if row['count_high'] <= 0:
        return np.nan
    p95 = row['p95']
    mx  = row['max_in_window']
    if pd.isna(p95) or p95 == 0:
        if pd.isna(mx) or mx == 0:
            return np.nan
        return np.inf
    return mx / p95

def compute_sev_low(row):
    if row['count_low'] <= 0:
        return np.nan
    p5 = row['p5']
    mn = row['min_in_window']
    if pd.isna(mn) or mn == 0:
        if pd.isna(p5) or p5 == 0:
            return np.nan
        return np.inf
    return p5 / mn

anomalies_df['severity_high'] = anomalies_df.apply(compute_sev_high, axis=1)
anomalies_df['severity_low']  = anomalies_df.apply(compute_sev_low, axis=1)

# 8) Format earliest_anom_ts as string in UTC+8 for compact display
anomalies_df['earliest_anom_ts'] = anomalies_df['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 9) Sort by (count_high+count_low) descending, then by max severity (either high or low) descending
anomalies_df['severity_sort'] = anomalies_df[['severity_high','severity_low']].fillna(0).max(axis=1)
anomalies_df = anomalies_df.sort_values(['anom_count','severity_sort'], ascending=[False, False])

# 10) Select requested columns and limit to top 20
out_cols = ['cmdb_id','kpi_name','count_high','count_low','count_in_window','earliest_anom_ts','p95','p5','max_in_window','min_in_window','severity_high','severity_low']
anomalies_top20 = anomalies_df.loc[:, out_cols].head(20).copy()

# Round numeric columns for compactness
for c in ['p95','p5','max_in_window','min_in_window','severity_high','severity_low']:
    if c in anomalies_top20.columns:
        anomalies_top20[c] = anomalies_top20[c].apply(lambda x: float(np.round(x,6)) if pd.notnull(x) and np.isfinite(x) else (np.inf if x==np.inf else x))

# 11) Summary info: total distinct pairs with anomaly and earliest anomaly timestamp overall
total_pairs_with_anom = anomalies_df.shape[0]
if total_pairs_with_anom > 0:
    # earliest_anom_ts in anomalies_df may be string; parse back to datetime with tz for min
    # But we earlier formatted to string; use anomalies_df['earliest_anom_ts'] before formatting if needed.
    # We still have earliest times in the merged earliest_per_pair; recompute overall earliest from anom_rows
    overall_earliest = anom_rows['dt'].min() if not anom_rows.empty else pd.NaT
    if pd.notna(overall_earliest):
        overall_earliest_str = overall_earliest.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        overall_earliest_str = ""
else:
    overall_earliest_str = ""
summary_line = f"total_pairs_with_anomaly={total_pairs_with_anom}, overall_earliest_anom_ts={overall_earliest_str}"

# Return compact table and summary line (kernel will display these variables)
anomalies_top20, summary_line

```
Out[2]:
```
```python
summary_text = (
    "Anomaly detection summary (window: 2022-03-20 22:00 to 22:30, UTC+8):\n\n"
    "- Using global P95 and P5 computed from the full metric series, then evaluating metric points inside the incident window.\n"
    f"- Distinct (cmdb_id,kpi_name) pairs with at least one anomaly in-window: 5278.\n"
    f"- Earliest anomaly timestamp across all pairs: 2022-03-20 22:00:00 (UTC+8).\n\n"
    "- Observations from the top results:\n"
    "  * Many of the top items are for the adservice component (e.g., app.grpc.rr, app.grpc.sr, app.http.count, app.http.rr, app.http.sr, several runtime/GC metrics).\n"
    "  * For those adservice entries shown in the top-20 table, count_high = 31 and count_low = 31 (count_in_window = 31), and severity_high = severity_low = 1.0. "
    "    This indicates the in-window values equal the computed P95 and P5 (no value exceeded P95 by >0 or dropped below P5), i.e., values are at the threshold.\n\n"
    "- Definitions used:\n"
    "  * count_high: number of in-window samples >= global P95\n"
    "  * count_low: number of in-window samples <= global P5\n"
    "  * severity_high: max_in_window / p95  (only meaningful if count_high>0)\n"
    "  * severity_low: p5 / min_in_window   (only meaningful if count_low>0)\n\n"
    "- Recommended next steps:\n"
    "  1) Inspect pairs with anom_count > 0 and severity > 1 to find true exceedances (values that surpass P95 by a margin).\n"
    "  2) For pairs with severity ≈ 1 and high counts (like many adservice metrics), check whether the series is constant or quantized so thresholds coincide with observed values.\n"
    "  3) Cross-correlate these anomalous metric pairs with trace/log anomalies (and with pod/node identities) to identify probable root causes.\n\n"
    "All full intermediate DataFrames remain in kernel variables (df_metric, thresholds_df, df_window, df_w_thr, anomalies_df, anomalies_top20) for further drill-down."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  count_high  count_low  count_in_window         earliest_anom_ts          p95           p5  max_in_window  min_in_window  severity_high  severity_low
2   adservice                                        app.grpc.rr          31         31               31  2022-03-20 22:00:00 CST        100.0        100.0          100.0          100.0            1.0           1.0
3   adservice                                        app.grpc.sr          31         31               31  2022-03-20 22:00:00 CST        100.0        100.0          100.0          100.0            1.0           1.0
4   adservice                                     app.http.count          31         31               31  2022-03-20 22:00:00 CST         24.0         24.0           24.0           24.0            1.0           1.0
6   adservice                                        app.http.rr          31         31               31  2022-03-20 22:00:00 CST        100.0        100.0          100.0          100.0            1.0           1.0
7   adservice                                        app.http.sr          31         31               31  2022-03-20 22:00:00 CST        100.0        100.0          100.0          100.0            1.0           1.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount          31         31               31  2022-03-20 22:00:00 CST       5162.0       5162.0         5162.0         5162.0            1.0           1.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...          31         31               31  2022-03-20 22:00:00 CST       5185.0       5185.0         5185.0         5185.0            1.0           1.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount          31         31               31  2022-03-20 22:00:00 CST         23.0         23.0           23.0           23.0            1.0           1.0
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...          31         31               31  2022-03-20 22:00:00 CST          1.0          1.0            1.0            1.0            1.0           1.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...          31         31               31  2022-03-20 22:00:00 CST          8.0          8.0            8.0            8.0            1.0           1.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...          31         31               31  2022-03-20 22:00:00 CST        480.0        480.0          480.0          480.0            1.0           1.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST          1.0          1.0            1.0            1.0            1.0           1.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST          1.0          1.0            1.0            1.0            1.0           1.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST         94.0         94.0           94.0           94.0            1.0           1.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST  539245384.0  539245384.0    539245384.0    539245384.0            1.0           1.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST          8.0          8.0            8.0            8.0            1.0           1.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST   23330816.0   23330816.0     23330816.0     23330816.0            1.0           1.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST   23330816.0   23330816.0     23330816.0     23330816.0            1.0           1.0
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST    3932160.0    3932160.0      3932160.0      3932160.0            1.0           1.0
29  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          31         31               31  2022-03-20 22:00:00 CST    3932160.0    3932160.0      3932160.0      3932160.0            1.0           1.0, 'total_pairs_with_anomaly=5278, overall_earliest_anom_ts=2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[3]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[3]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric; ensure dt exists and timezone is Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = df_metric['dt'].dt.tz_localize(pytz.UTC).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# Incident window (inclusive)
start = pd.Timestamp('2022-03-20 22:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 22:30:00', tz=tz)

# 1) Compute global thresholds p95 and p5 per (cmdb_id, kpi_name) using full series
thresholds_df = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile([0.95,0.05]).unstack(level=-1).reset_index()
# After unstack quantile, columns may be [0.95,0.05]; rename properly
# If columns are floats, map them
col_map = {}
for c in thresholds_df.columns:
    if isinstance(c, float):
        if np.isclose(c, 0.95):
            col_map[c] = 'p95'
        elif np.isclose(c, 0.05):
            col_map[c] = 'p5'
thresholds_df = thresholds_df.rename(columns=col_map)
# If quantile columns named differently (e.g., 0.95,0.05) ensure p95 and p5 exist
if 'p95' not in thresholds_df.columns or 'p5' not in thresholds_df.columns:
    # safer explicit compute
    thr_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
    thr_p5  = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index(name='p5')
    thresholds_df = thr_p95.merge(thr_p5, on=['cmdb_id','kpi_name'])

# 2) Filter df_metric to incident window
df_window = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# 3) Merge thresholds into windowed rows
df_w_thr = df_window.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left')
# Drop rows lacking thresholds
df_w_thr = df_w_thr.dropna(subset=['p95','p5'])

# 4) Flag anomalous rows (high or low)
df_w_thr['is_high'] = df_w_thr['value'] >= df_w_thr['p95']
df_w_thr['is_low']  = df_w_thr['value'] <= df_w_thr['p5']
df_w_thr['is_anom'] = df_w_thr['is_high'] | df_w_thr['is_low']

# 5) Aggregate per (cmdb_id,kpi_name): basic stats
grp = df_w_thr.groupby(['cmdb_id','kpi_name'])
pair_stats = grp.agg(
    count_high = ('is_high','sum'),
    count_low  = ('is_low','sum'),
    count_in_window = ('value','size'),
    max_in_window = ('value','max'),
    min_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
).reset_index()

# earliest anomalous timestamp per pair
anom_rows = df_w_thr[df_w_thr['is_anom']]
if not anom_rows.empty:
    earliest_per_pair = anom_rows.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index(name='earliest_anom_ts')
    pair_stats = pair_stats.merge(earliest_per_pair, on=['cmdb_id','kpi_name'], how='left')
else:
    pair_stats['earliest_anom_ts'] = pd.NaT

# 6) Compute severity_high and severity_low
def sev_high(r):
    if r['count_high'] <= 0:
        return np.nan
    p95 = r['p95']
    mx = r['max_in_window']
    if pd.isna(p95) or p95 == 0:
        return np.inf if mx>0 else np.nan
    return mx / p95

def sev_low(r):
    if r['count_low'] <= 0:
        return np.nan
    p5 = r['p5']
    mn = r['min_in_window']
    if pd.isna(mn) or mn == 0:
        return np.inf if p5>0 else np.nan
    return p5 / mn

pair_stats['severity_high'] = pair_stats.apply(sev_high, axis=1)
pair_stats['severity_low']  = pair_stats.apply(sev_low, axis=1)

# 7) Compute longest_consecutive_anom_minutes per pair
# We'll compute runs on anomalous rows per pair: consecutive minutes where dt difference == 60s
def longest_consec_minutes(sub):
    # sub is df_w_thr subset for one (cmdb_id,kpi_name) with only anomalous rows
    if sub.empty:
        return 0
    s = sub.sort_values('dt')['dt']
    diffs = s.diff().dt.total_seconds().fillna(0)
    # start new group when diff != 60
    new_group = (diffs != 60).cumsum()
    run_lengths = new_group.groupby(new_group).size()
    return int(run_lengths.max())

# Build anomalous grouped frame for iteration
anom_grouped = anom_rows.groupby(['cmdb_id','kpi_name'])

# compute longest run and attach
pair_stats['longest_consecutive_anom_minutes'] = 0
# iterate only pairs that have anomalies to save time
for (cmdb,kpi), group in anom_grouped:
    mask = (pair_stats['cmdb_id']==cmdb) & (pair_stats['kpi_name']==kpi)
    pair_stats.loc[mask, 'longest_consecutive_anom_minutes'] = longest_consec_minutes(group)

# 8) Keep only pairs where severity_high>1 OR severity_low>1
breached_pairs = pair_stats[
    ((pair_stats['severity_high'] > 1.0) & (pair_stats['count_high']>0)) |
    ((pair_stats['severity_low']  > 1.0) & (pair_stats['count_low']>0))
].copy()

# Format earliest_anom_ts as string for compact display
breached_pairs['earliest_anom_ts_str'] = breached_pairs['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 9) Aggregate breached pairs by cmdb_id
# total_breach_points = sum(count_high + count_low)
breached_pairs['breach_points'] = breached_pairs['count_high'] + breached_pairs['count_low']
# severity overall per pair
breached_pairs['max_severity_pair'] = breached_pairs[['severity_high','severity_low']].fillna(0).max(axis=1)

# For top_kpi_example per cmdb_id, pick kpi with max_severity_pair (tie-breaker: highest breach_points)
bk = breached_pairs.copy()
# sort so first per group is desired top kpi
bk = bk.sort_values(['cmdb_id','max_severity_pair','breach_points'], ascending=[True, False, False])
top_kpi_per_cmdb = bk.groupby('cmdb_id').first().reset_index()[['cmdb_id','kpi_name','max_severity_pair','breach_points','earliest_anom_ts_str']]

agg = breached_pairs.groupby('cmdb_id').agg(
    num_breached_kpis = ('kpi_name','nunique'),
    total_breach_points = ('breach_points','sum'),
    earliest_breach_ts = ('earliest_anom_ts','min'),
    max_severity_overall = ('max_severity_pair','max')
).reset_index()

# attach top_kpi_example
agg = agg.merge(top_kpi_per_cmdb[['cmdb_id','kpi_name','max_severity_pair']], on='cmdb_id', how='left')
agg = agg.rename(columns={'kpi_name':'top_kpi_example', 'max_severity_pair':'top_kpi_severity'})

# format earliest_breach_ts string
agg['earliest_breach_ts'] = agg['earliest_breach_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 10) Order and limit to top 20 rows by num_breached_kpis desc, then max_severity_overall desc
agg = agg.sort_values(['num_breached_kpis','max_severity_overall'], ascending=[False, False]).reset_index(drop=True)
breaches_by_cmdb_top20 = agg.loc[:, ['cmdb_id','num_breached_kpis','total_breach_points','earliest_breach_ts','max_severity_overall','top_kpi_example']].head(20).copy()

# Round numeric severities for compact display
breaches_by_cmdb_top20['max_severity_overall'] = breaches_by_cmdb_top20['max_severity_overall'].apply(lambda x: float(np.round(x,6)) if pd.notnull(x) and np.isfinite(x) else x)

# 11) Summary values
total_number_of_breached_pairs = breached_pairs.shape[0]
if total_number_of_breached_pairs > 0:
    overall_earliest_breach_ts = breached_pairs['earliest_anom_ts'].min()
    overall_earliest_breach_ts = overall_earliest_breach_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    overall_earliest_breach_ts = ""

# Expose main outputs (compact)
breaches_by_cmdb_top20, total_number_of_breached_pairs, overall_earliest_breach_ts

```
Out[4]:
```
```python
summary = (
    "Anomaly summary (window: 2022-03-20 22:00–22:30 UTC+8):\n\n"
    "- Total distinct breached (cmdb_id,kpi_name) pairs: 1,484.\n"
    "- Earliest breach observed: 2022-03-20 22:00:00 (UTC+8).\n\n"
    "- Most affected components (top 20 shown):\n"
    "  * frontend-0 (66 breached KPIs, 277 breach points) — top example KPI: mesh.source.frontend.checkoutservice.istio_req..., max severity = inf\n"
    "  * frontend-2 (62 breached KPIs, 275 breach points) — top example KPI: container.node-5.container_network_transmit_MB..., max severity ≈ 2.532\n"
    "  * frontend-1 (61 breached KPIs, 300 breach points) — top example KPI: mesh.destination.unknown.frontend.istio_reques..., max severity = inf\n"
    "  * frontend2-0, checkoutservice-1, checkoutservice-2, productcatalogservice (pods), node-5, node-6, node-1, node-4, node-3, cartservice pods, currencyservice2-0, adservice2, currencyservice-2 also appear with substantial breached KPI counts.\n\n"
    "- Notes on severity values:\n"
    "  * Several top entries show severity = inf, which indicates a division by zero in the threshold (p95 or min_in_window = 0) or extremely small baseline — this typically means the baseline threshold is zero while observed values are >0 (true spike), or min_in_window is zero for low-severity calculation.\n"
    "  * Some nodes/pods exhibit very high numeric severity (e.g., node-1 has a high severity for system.net.tcp.retrans_segs, node-5 shows high system.net.udp out_datagrams severity); these are strong signals for network or I/O issues.\n\n"
    "- Immediate recommended next steps:\n"
    "  1) Drill into the top cmdb_ids above (start with frontend-0/1/2 and node-5/node-1) and inspect the top_kpi_example metrics listed for each — these are the KPIs that produced the maximum severity.\n"
    "  2) For entries with severity = inf, examine raw series and confirm whether thresholds are zero (and whether observed values represent real spikes).\n"
    "  3) Correlate these metric breaches with traces and logs (trace.csv and log.csv) to find request-path or error correlates and narrow root cause (network retries, retransmissions, I/O latency, or CPU throttling are likely candidates based on top KPIs).\n\n"
    "If you want, I can now: (A) fetch the top_kpi_example time series for any specific cmdb_id, (B) list all KPIs with severity > 2, or (C) correlate these breaches with trace/log anomalies."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  num_breached_kpis  total_breach_points       earliest_breach_ts  max_severity_overall                                    top_kpi_example
0                frontend-0                 66                  277  2022-03-20 22:00:00 CST                   inf  mesh.source.frontend.checkoutservice.istio_req...
1                frontend-2                 62                  275  2022-03-20 22:00:00 CST              2.532373  container.node-5.container_network_transmit_MB...
2                frontend-1                 61                  300  2022-03-20 22:00:00 CST                   inf  mesh.destination.unknown.frontend.istio_reques...
3               frontend2-0                 59                  265  2022-03-20 22:00:00 CST                   inf  mesh.destination.unknown.frontend2.istio_reque...
4         checkoutservice-1                 48                  484  2022-03-20 22:00:00 CST                   inf  mesh.destination.frontend.checkoutservice.isti...
5         checkoutservice-2                 44                  456  2022-03-20 22:00:00 CST                   inf  container.node-5.container_memory_failures.con...
6   productcatalogservice-1                 44                  348  2022-03-20 22:00:00 CST                   inf  mesh.source.productcatalogservice.jaeger-colle...
7   productcatalogservice-2                 41                  348  2022-03-20 22:00:00 CST              3.000000  mesh.destination.checkoutservice.productcatalo...
8   productcatalogservice-0                 39                  315  2022-03-20 22:00:00 CST                   inf  container.node-6.container_memory_failures.con...
9                    node-5                 39                  473  2022-03-20 22:00:00 CST              9.683784                       system.net.udp.out_datagrams
10        checkoutservice-0                 38                  188  2022-03-20 22:00:00 CST                   inf  mesh.source.checkoutservice.paymentservice.ist...
11                   node-6                 37                  260  2022-03-20 22:00:00 CST              2.074468                                  system.io.w_await
12                   node-1                 35                  210  2022-03-20 22:00:00 CST             25.800000                        system.net.tcp.retrans_segs
13                   node-4                 34                  347  2022-03-20 22:00:00 CST                   inf                                 system.io.avg_q_sz
14                   node-3                 34                  193  2022-03-20 22:00:00 CST              4.888889                                    system.io.svctm
15            cartservice-1                 33                  296  2022-03-20 22:00:00 CST                   inf  container.node-6.container_cpu_cfs_throttled_p...
16            cartservice-2                 33                  285  2022-03-20 22:00:00 CST                   inf  container.node-6.container_cpu_cfs_throttled_p...
17       currencyservice2-0                 32                  171  2022-03-20 22:00:00 CST              7.823466  container.node-6.container_cpu_cfs_throttled_s...
18               adservice2                 32                  172  2022-03-20 22:00:00 CST              1.275326  runtime.java_lang_OperatingSystem_SystemLoadAv...
19        currencyservice-2                 31                  148  2022-03-20 22:00:00 CST              5.770136  container.node-6.container_network_receive_MB...., 1484, '2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and thresholds_df if present; ensure dt timezone is Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = df_metric['dt'].dt.tz_localize(pytz.UTC).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# Incident window boundaries (inclusive)
start = pd.Timestamp('2022-03-20 22:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 22:30:00', tz=tz)

# 1) Select node-1 KPIs whose kpi_name contains any of the keywords
keywords = ['net','tcp','udp','retrans','packets','packet']
pattern = '|'.join(keywords)
node1_all = df_metric[(df_metric['cmdb_id']=='node-1') & df_metric['kpi_name'].str.contains(pattern, case=False, na=False)].copy()

# Ensure global thresholds (p95,p5) exist for these pairs. Use thresholds_df if available; otherwise compute for all and reuse.
if 'thresholds_df' not in globals() or thresholds_df is None:
    thr_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
    thr_p5  = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index(name='p5')
    thresholds_df = thr_p95.merge(thr_p5, on=['cmdb_id','kpi_name'])

# Subset thresholds to node-1 KPIs (but keep full thresholds_df in memory)
node1_kpis = thresholds_df[thresholds_df['cmdb_id']=='node-1'][['cmdb_id','kpi_name','p95','p5']].copy()

# If node1_all is empty, prepare empty outputs
if node1_all.empty or node1_kpis.empty:
    node1_top20 = pd.DataFrame(columns=['kpi_name','count_in_window','count_high','count_low','earliest_anom_ts','p95','p5','max_in_window','min_in_window','severity_high','severity_low','longest_consecutive_anom_minutes'])
    summary_line = "no node-1 network-related KPIs found in df_metric for the given keywords/window"
else:
    # 2) For each (node-1, kpi_name): process windowed points
    # Merge thresholds into node1_all to allow row-level anomaly flags
    node1_all = node1_all.merge(node1_kpis, on=['cmdb_id','kpi_name'], how='left').dropna(subset=['p95','p5'])

    # Filter to incident window
    node1_window = node1_all[(node1_all['dt'] >= start) & (node1_all['dt'] <= end)].copy()

    # If no points in window, produce empty outputs
    if node1_window.empty:
        node1_top20 = pd.DataFrame(columns=['kpi_name','count_in_window','count_high','count_low','earliest_anom_ts','p95','p5','max_in_window','min_in_window','severity_high','severity_low','longest_consecutive_anom_minutes'])
        summary_line = "no node-1 points in the incident window"
    else:
        # Flag anomalies per row (high or low)
        node1_window['is_high'] = node1_window['value'] >= node1_window['p95']
        node1_window['is_low']  = node1_window['value'] <= node1_window['p5']
        node1_window['is_anom'] = node1_window['is_high'] | node1_window['is_low']

        # Aggregate basic stats per kpi
        g = node1_window.groupby('kpi_name')
        stats = g.agg(
            count_in_window = ('value','size'),
            count_high = ('is_high','sum'),
            count_low  = ('is_low','sum'),
            max_in_window = ('value','max'),
            min_in_window = ('value','min'),
            p95 = ('p95','first'),
            p5  = ('p5','first')
        ).reset_index()

        # earliest anomalous timestamp per kpi
        anom_rows = node1_window[node1_window['is_anom']]
        if not anom_rows.empty:
            earliest = anom_rows.groupby('kpi_name')['dt'].min().reset_index(name='earliest_anom_ts')
            stats = stats.merge(earliest, on='kpi_name', how='left')
        else:
            stats['earliest_anom_ts'] = pd.NaT

        # compute severity metrics
        def sev_high_row(r):
            if r['count_high'] <= 0:
                return np.nan
            p95 = r['p95']
            mx = r['max_in_window']
            if pd.isna(p95) or p95 == 0:
                return np.inf if (mx>0) else np.nan
            return mx / p95

        def sev_low_row(r):
            if r['count_low'] <= 0:
                return np.nan
            p5 = r['p5']
            mn = r['min_in_window']
            if pd.isna(mn) or mn == 0:
                return np.inf if (p5>0) else np.nan
            return p5 / mn

        stats['severity_high'] = stats.apply(sev_high_row, axis=1)
        stats['severity_low']  = stats.apply(sev_low_row, axis=1)

        # compute longest consecutive anomalous minutes per kpi
        def longest_consec(sub):
            # sub: subset of node1_window for one kpi with anomalous rows only
            if sub.empty:
                return 0
            s = sub.sort_values('dt')['dt']
            diffs = s.diff().dt.total_seconds().fillna(0)
            groups = (diffs != 60).cumsum()
            run_lengths = groups.groupby(groups).size()
            return int(run_lengths.max())

        # compute for each kpi
        longest_map = {}
        for kpi, grp_df in anom_rows.groupby('kpi_name'):
            longest_map[kpi] = longest_consec(grp_df)

        stats['longest_consecutive_anom_minutes'] = stats['kpi_name'].map(longest_map).fillna(0).astype(int)

        # Keep only KPIs with true breach severity > 1 for either side
        stats['max_severity'] = stats[['severity_high','severity_low']].fillna(0).max(axis=1)
        breached_stats = stats[((stats['severity_high']>1.0) & (stats['count_high']>0)) | ((stats['severity_low']>1.0) & (stats['count_low']>0))].copy()

        # Format earliest_anom_ts column
        breached_stats['earliest_anom_ts'] = breached_stats['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

        # 3) Sort and prepare top 20
        breached_stats = breached_stats.sort_values(['max_severity','longest_consecutive_anom_minutes'], ascending=[False, False]).reset_index(drop=True)
        node1_top20 = breached_stats.loc[:, ['kpi_name','count_in_window','count_high','count_low','earliest_anom_ts','p95','p5','max_in_window','min_in_window','severity_high','severity_low','longest_consecutive_anom_minutes']].head(20).copy()

        # Round numeric columns for compactness
        for c in ['p95','p5','max_in_window','min_in_window','severity_high','severity_low']:
            if c in node1_top20.columns:
                node1_top20[c] = node1_top20[c].apply(lambda x: float(np.round(x,6)) if pd.notnull(x) and np.isfinite(x) else x)

        # 4) Summary line
        exists_long_run = (breached_stats['longest_consecutive_anom_minutes'] >= 2).any()
        total_breached_kpis = breached_stats.shape[0]
        if total_breached_kpis > 0:
            overall_earliest = pd.to_datetime(breached_stats['earliest_anom_ts'].dropna().min())
            # earliest_anom_ts strings are in local tz; convert to formatted string (they already include TZ)
            overall_earliest_str = breached_stats['earliest_anom_ts'].dropna().min()
            # KPI with max severity
            top_idx = breached_stats['max_severity'].idxmax()
            top_kpi = breached_stats.loc[top_idx, 'kpi_name']
            top_sev = breached_stats.loc[top_idx, 'max_severity']
            top_sev = float(np.round(top_sev,6)) if pd.notnull(top_sev) and np.isfinite(top_sev) else top_sev
        else:
            overall_earliest_str = ""
            top_kpi = ""
            top_sev = np.nan

        summary_line = (
            f"exists_long_run>={{True}} if any KPI has >=2 consecutive anomalous minutes: {bool(exists_long_run)}; "
            f"overall_earliest_anom_ts={overall_earliest_str}; "
            f"top_kpi_by_severity={top_kpi}; top_severity={top_sev}"
        )

# Return compact results
node1_top20, summary_line

```
Out[5]:
```
Summary of node-1 network-related metric inspection (window 2022-03-20 22:00–22:30 UTC+8):

Key findings
- Earliest anomalous point on node-1: 2022-03-20 22:00:00 (UTC+8).
- At least one node-1 KPI shows a consecutive anomalous run >= 2 minutes → True (indicates a likely fault).
- The KPI with the largest severity is system.net.tcp.retrans_segs (severity = 25.8).

Top node-1 network KPIs with true breaches (compact):
- system.net.tcp.retrans_segs
  - in-window samples: 31; count_high=2; count_low=21
  - earliest anomaly: 2022-03-20 22:00:00
  - p95=0.20, p5=0.00; max_in_window=5.16, min_in_window=0.00
  - severity_high = 25.8 (max/p95); severity_low = NaN (p5 or min = 0)
  - longest consecutive anomalous minutes = 8

- system.net.tcp.in_segs
  - in-window: 31; count_high=3; count_low=3
  - earliest anomaly: 2022-03-20 22:08:00
  - p95≈3957.24, p5≈902.47; max≈12428.45, min≈788.97
  - severity_high ≈ 3.14; severity_low ≈ 1.1439
  - longest consecutive anomalous minutes = 4

- system.net.bytes_rcvd
  - in-window: 31; count_high=2; count_low=4
  - earliest anomaly: 2022-03-20 22:09:00
  - p95≈3,440,410.75, p5≈727,810.38; max≈10,022,455.00, min≈561,204.56
  - severity_high ≈ 2.913; severity_low ≈ 1.297
  - longest consecutive anomalous minutes = 3

- system.net.packets_in.count
  - in-window: 31; count_high=2; count_low=4
  - earliest anomaly: 2022-03-20 22:09:00
  - p95≈2525.65, p5≈653.51; max≈6928.66, min≈544.52
  - severity_high ≈ 2.743; severity_low ≈ 1.200
  - longest consecutive anomalous minutes = 3

- system.net.packets_out.count
  - in-window: 31; count_high=2; count_low=3
  - earliest anomaly: 2022-03-20 22:09:00
  - p95≈634.11, p5≈320.02; max≈1342.05, min≈312.01
  - severity_high ≈ 2.116; severity_low ≈ 1.0257
  - longest consecutive anomalous minutes = 2

- system.net.bytes_sent
  - in-window: 31; count_high=3; count_low=3
  - earliest anomaly: 2022-03-20 22:03:00
  - p95≈1,851,593.25, p5≈654,763.94; max≈2,210,438.50, min≈344,177.34
  - severity_high ≈ 1.194; severity_low ≈ 1.902
  - longest consecutive anomalous minutes = 3

- system.net.udp.in_datagrams
  - in-window: 31; count_high=1; count_low=3
  - earliest anomaly: 2022-03-20 22:00:00
  - p95≈661.12, p5≈443.78; max≈959.11, min≈353.83
  - severity_high ≈ 1.451; severity_low ≈ 1.254
  - longest consecutive anomalous minutes = 3

- system.net.tcp.out_segs
  - in-window: 31; count_high=2; count_low=1
  - earliest anomaly: 2022-03-20 22:03:00
  - p95≈11,667.82, p5≈3,445.67; max≈16,297.72, min≈3,298.85
  - severity_high ≈ 1.397; severity_low ≈ 1.045
  - longest consecutive anomalous minutes = 1

- system.net.udp.out_datagrams
  - in-window: 31; count_high=4; count_low=5
  - earliest anomaly: 2022-03-20 22:00:00
  - p95≈1.08, p5≈0.98; max≈1.18, min≈0.97
  - severity_high ≈ 1.093; severity_low ≈ 1.010
  - longest consecutive anomalous minutes = 5

Interpretation and recommended next steps
- Strong signal of network problems on node-1:
  - Very high TCP retransmissions (system.net.tcp.retrans_segs) with severity ≈ 25.8 and an 8-minute consecutive anomalous run — this strongly indicates packet loss/retransmission issues on node-1’s network path or NIC.
  - Elevated TCP in/out segments, bytes received/sent, and packet counts with multi-minute anomalous runs — consistent with network congestion, retransmits, or bursts.
  - UDP datagram anomalies (out_datagrams in particular) also show multi-minute runs, suggesting broader packet-level anomalies beyond TCP-only issues.
- Actions to take immediately:
  1) Inspect node-1 network interface stats at the host level (link errors, drops, NIC interrupts, driver messages) and node system logs for errors around 22:00–22:10.  
  2) Check network path (switch/route) and any recent changes or spikes in traffic that could cause congestion.  
  3) Correlate these metric breaches with traces/logs for services hosted on node-1 to see impacted requests or retries.  
  4) If retransmissions remain high, consider isolating the node (or moving pods) until network root cause is resolved.

If you want, I can now fetch the time series for any one of these KPIs (e.g., system.net.tcp.retrans_segs) across the window for closer inspection, or correlate node-1 breaches with trace/log events.

The original code execution output of IPython Kernel is also provided below for reference:

(                       kpi_name  count_in_window  count_high  count_low         earliest_anom_ts         p95         p5  max_in_window  min_in_window  severity_high  severity_low  longest_consecutive_anom_minutes
0   system.net.tcp.retrans_segs               31           2         21  2022-03-20 22:00:00 CST        0.20       0.00           5.16           0.00      25.800000           NaN                                 8
1        system.net.tcp.in_segs               31           3          3  2022-03-20 22:08:00 CST     3957.24     902.47       12428.45         788.97       3.140686      1.143858                                 4
2         system.net.bytes_rcvd               31           2          4  2022-03-20 22:09:00 CST  3440410.75  727810.38    10022455.00      561204.56       2.913157      1.296872                                 3
3   system.net.packets_in.count               31           2          4  2022-03-20 22:09:00 CST     2525.65     653.51        6928.66         544.52       2.743318      1.200158                                 3
4  system.net.packets_out.count               31           2          3  2022-03-20 22:09:00 CST      634.11     320.02        1342.05         312.01       2.116431      1.025672                                 2
5         system.net.bytes_sent               31           3          3  2022-03-20 22:03:00 CST  1851593.25  654763.94     2210438.50      344177.34       1.193803      1.902403                                 3
6   system.net.udp.in_datagrams               31           1          3  2022-03-20 22:00:00 CST      661.12     443.78         959.11         353.83       1.450735      1.254218                                 3
7       system.net.tcp.out_segs               31           2          1  2022-03-20 22:03:00 CST    11667.82    3445.67       16297.72        3298.85       1.396809      1.044506                                 1
8  system.net.udp.out_datagrams               31           4          5  2022-03-20 22:00:00 CST        1.08       0.98           1.18           0.97       1.092593      1.010309                                 5, 'exists_long_run>={True} if any KPI has >=2 consecutive anomalous minutes: True; overall_earliest_anom_ts=2022-03-20 22:00:00 CST; top_kpi_by_severity=system.net.tcp.retrans_segs; top_severity=25.8')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Ensure df_trace dt column exists and is timezone-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_trace['dt'].dt.tz is None:
        df_trace['dt'] = df_trace['dt'].dt.tz_localize(pytz.UTC).dt.tz_convert(tz)
    else:
        df_trace['dt'] = df_trace['dt'].dt.tz_convert(tz)

# Use the previously computed top cmdb_ids list (from breaches_by_cmdb_top20 if available)
if 'breaches_by_cmdb_top20' in globals() and not breaches_by_cmdb_top20.empty:
    target_cmdbs = breaches_by_cmdb_top20['cmdb_id'].astype(str).tolist()
else:
    # Fallback: take top 20 distinct cmdb_ids from df_trace
    target_cmdbs = df_trace['cmdb_id'].dropna().unique()[:20].tolist()

# Trace metrics of interest
trace_metrics = ['trace.self.duration_mean', 'trace.self.error_rate']

# 1) Compute global p95 per (cmdb_id, trace_name) using full series
thr_trace_p95 = df_trace[df_trace['trace_name'].isin(trace_metrics)].groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index(name='p95')

# 2) Filter df_trace to target cmdb_ids, metrics, and incident window
start = pd.Timestamp('2022-03-20 22:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 22:30:00', tz=tz)
trace_win = df_trace[
    (df_trace['cmdb_id'].isin(target_cmdbs)) &
    (df_trace['trace_name'].isin(trace_metrics)) &
    (df_trace['dt'] >= start) & (df_trace['dt'] <= end)
].copy()

# Merge p95 thresholds (global) into windowed rows
trace_win = trace_win.merge(thr_trace_p95, on=['cmdb_id','trace_name'], how='left')
# Drop rows without p95 (no global threshold)
trace_win = trace_win.dropna(subset=['p95'])

# Flag high (>= p95)
trace_win['is_high'] = trace_win['value'] >= trace_win['p95']

# 2) Aggregate per (cmdb_id, trace_name)
grp = trace_win.groupby(['cmdb_id','trace_name'])
agg = grp.agg(
    count_in_window = ('value','size'),
    count_high = ('is_high','sum'),
    earliest_high_dt = ('dt', lambda s: s[s >= s.min()].loc[trace_win.loc[s.index,'is_high']].min() if True else pd.NaT),  # placeholder, replaced below
    max_in_window = ('value','max'),
    mean_in_window = ('value','mean'),
    p95 = ('p95','first')
).reset_index()

# Compute earliest_high_dt properly using the filtered dataframe (avoid lambda complexity)
earliest_high = trace_win[trace_win['is_high']].groupby(['cmdb_id','trace_name'])['dt'].min().reset_index(name='earliest_anom_ts')
agg = agg.drop(columns=['earliest_high_dt'])
agg = agg.merge(earliest_high, on=['cmdb_id','trace_name'], how='left')

# Compute severity = max_in_window / p95 (only meaningful if count_high>0)
def compute_sev(row):
    if row['count_high'] <= 0:
        return np.nan
    p95 = row['p95']
    mx = row['max_in_window']
    if pd.isna(p95) or p95 == 0:
        return np.inf if mx>0 else np.nan
    return mx / p95

agg['severity'] = agg.apply(compute_sev, axis=1)

# Format earliest_anom_ts to string
agg['earliest_anom_ts'] = agg['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Keep requested columns and rename trace_name -> trace_metric
agg = agg.rename(columns={'trace_name':'trace_metric'})

# 3) Order rows by count_high desc then severity desc, limit top 20
trace_top_table = agg.sort_values(['count_high','severity'], ascending=[False, False]).reset_index(drop=True)
trace_top_table = trace_top_table.loc[:, ['cmdb_id','trace_metric','count_in_window','count_high','earliest_anom_ts','p95','max_in_window','mean_in_window','severity']].head(20).copy()

# Round numeric columns for compactness
for c in ['p95','max_in_window','mean_in_window','severity']:
    if c in trace_top_table.columns:
        trace_top_table[c] = trace_top_table[c].apply(lambda x: float(np.round(x,6)) if pd.notnull(x) and np.isfinite(x) else x)

# 4) Summary: how many cmdb_ids show trace anomalies with earliest_anom_ts <= 22:05 and earliest overall trace anomaly among them
# Compute earliest per cmdb_id across the two trace metrics
if not agg.empty:
    per_cmdb_earliest = agg.dropna(subset=['earliest_anom_ts']).copy()
    # convert back to Timestamp for comparison
    per_cmdb_earliest['earliest_dt'] = pd.to_datetime(per_cmdb_earliest['earliest_anom_ts']).dt.tz_localize(None)
    # Need to compare with 22:05 in local tz: create naive timestamp consistent with string conversion above
    cutoff = pd.Timestamp('2022-03-20 22:05:00')  # naive, matches formatting
    # Count distinct cmdb_ids with earliest_anom_ts <= cutoff
    cmdbs_with_early = per_cmdb_earliest[per_cmdb_earliest['earliest_dt'] <= cutoff]['cmdb_id'].unique()
    num_cmdbs_early = len(cmdbs_with_early)
    # overall earliest trace anomaly among these target rows
    overall_earliest_dt = per_cmdb_earliest['earliest_dt'].min()
    overall_earliest_str = per_cmdb_earliest.loc[per_cmdb_earliest['earliest_dt'].idxmin(), 'earliest_anom_ts']
else:
    num_cmdbs_early = 0
    overall_earliest_str = ""

summary_line = (
    f"num_cmdbs_with_trace_anom_<=22:05:00={num_cmdbs_early}, overall_earliest_trace_anom_ts={overall_earliest_str}"
)

# Return compact table and summary
trace_top_table, summary_line

```
Out[6]:
```
Summary of trace inspection for the top-breached components (window 2022-03-20 22:00–22:30 UTC+8):

Key overall points
- Many components show trace.self.error_rate rows flagged as "high" in-window, but their global p95 = 0 and max_in_window = 0.0. That makes those "high" flags not meaningful (they result from p95==0 and values==0). Treat trace.error_rate results with p95==0 as non-actionable unless you confirm nonzero error-rate values.
- Trace.duration_mean yields meaningful breaches for several components. These represent real latency increases relative to the full-series P95.

Top trace.duration_mean anomalies (most important)
- cartservice-1: count_in_window=30, count_high=2, earliest anomaly 2022-03-20 22:00:00, p95≈0.000040, max≈0.000158, severity ≈ 3.94 (largest ratio of max/p95).
- checkoutservice-2: count_in_window=25, count_high=2, earliest anomaly 2022-03-20 22:05:00, p95≈0.008533, max≈0.021902, severity ≈ 2.57.
- cartservice-2: count_in_window=30, count_high=2, earliest anomaly 2022-03-20 22:00:00, p95≈0.000046, max≈0.000105, severity ≈ 2.28.
- checkoutservice-1: count_in_window=27, count_high=2, earliest anomaly 2022-03-20 22:04:00, p95≈0.008319, max≈0.009883, severity ≈ 1.19.
- checkoutservice-0: count_in_window=27, count_high=2, earliest anomaly 2022-03-20 22:03:00, p95≈0.009488, max≈0.010724, severity ≈ 1.13.
- frontend2-0, frontend-2, frontend-0, frontend-1 also show small-but-above-threshold duration increases (severity ≈ 1.01–1.09) with earliest anomalies between 22:00 and 22:25.

Summary numbers
- Number of target cmdb_ids with trace anomalies occurring by 22:05:00: 9.
- Overall earliest trace anomaly among the inspected set: 2022-03-20 22:00:00 (UTC+8).

Interpretation and recommended next steps
1. Ignore the trace.self.error_rate "breaches" where p95==0 and max==0 — they are not actionable. Focus on trace.self.duration_mean breaches.
2. Priority targets: cartservice (both pods) and checkoutservice pods—these show the largest relative latency spikes (highest severity ratios) and earliest anomalies (22:00–22:05).
3. Correlate these duration spikes with the node-level network anomalies you already found on node-1 (notably high TCP retransmissions and packet anomalies around 22:00). That correlation (latency + network retransmits) indicates the service latency was likely caused by degraded network on nodes hosting these services.
4. Next actions: inspect request traces for impacted services for error/retry patterns, check pod placement (which pods were on node-1), and validate network health (NIC errors, switch logs, routing) for the timeframe 22:00–22:10.

If you want, I can (a) show the duration_mean time series for any specific pod (e.g., cartservice-1), or (b) cross-correlate these specific pod traces with node-1 metric anomalies and logs.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id              trace_metric  count_in_window  count_high         earliest_anom_ts       p95  max_in_window  mean_in_window  severity
0       cartservice-1     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
1       cartservice-2     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
2          frontend-0     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
3          frontend-1     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
4          frontend-2     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
5         frontend2-0     trace.self.error_rate               30          30  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
6   checkoutservice-0     trace.self.error_rate               27          27  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
7   checkoutservice-1     trace.self.error_rate               27          27  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
8   checkoutservice-2     trace.self.error_rate               25          25  2022-03-20 22:00:00 CST  0.000000       0.000000        0.000000       NaN
9       cartservice-1  trace.self.duration_mean               30           2  2022-03-20 22:00:00 CST  0.000040       0.000158        0.000008  3.940150
10  checkoutservice-2  trace.self.duration_mean               25           2  2022-03-20 22:05:00 CST  0.008533       0.021902        0.005563  2.566621
11      cartservice-2  trace.self.duration_mean               30           2  2022-03-20 22:00:00 CST  0.000046       0.000105        0.000006  2.277657
12  checkoutservice-1  trace.self.duration_mean               27           2  2022-03-20 22:04:00 CST  0.008319       0.009883        0.005325  1.188018
13  checkoutservice-0  trace.self.duration_mean               27           2  2022-03-20 22:03:00 CST  0.009488       0.010724        0.006052  1.130306
14        frontend2-0  trace.self.duration_mean               30           2  2022-03-20 22:00:00 CST  0.005231       0.005717        0.004792  1.092845
15         frontend-2  trace.self.duration_mean               30           2  2022-03-20 22:25:00 CST  0.005360       0.005639        0.004967  1.052033
16         frontend-0  trace.self.duration_mean               30           2  2022-03-20 22:00:00 CST  0.005279       0.005358        0.004848  1.014927
17         frontend-1  trace.self.duration_mean               30           2  2022-03-20 22:02:00 CST  0.356887       0.357826        0.062433  1.002631, 'num_cmdbs_with_trace_anom_<=22:05:00=9, overall_earliest_trace_anom_ts=2022-03-20 22:00:00 CST')```
```

In [ ]:
# In[7]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[7]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse df_metric already loaded in kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure dt exists and is timezone-aware (UTC+8)
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted to Asia/Shanghai
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = pd.to_datetime(df_metric['dt'], utc=True).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# 1) Find rows where kpi_name contains 'container.node-1' (case-insensitive)
mask_node1 = df_metric['kpi_name'].str.contains('container.node-1', case=False, na=False)
df_node1_map = df_metric[mask_node1].copy()

# 2) For each cmdb_id, count distinct kpi_name entries containing 'container.node-1'
mapping_counts = (
    df_node1_map.groupby('cmdb_id')['kpi_name']
    .nunique()
    .reset_index(name='metric_count')
    .sort_values('metric_count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# 3) Summary values
total_unique_cmdbs = int(df_node1_map['cmdb_id'].nunique())
if not df_node1_map.empty:
    earliest_metric_dt = df_node1_map['dt'].min()
    earliest_metric_str = earliest_metric_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    earliest_metric_str = ""

summary_line = f"total_unique_cmdb_ids={total_unique_cmdbs}, earliest_metric_timestamp_for_node-1_mappings={earliest_metric_str}"

# Return compact table (mapping_counts) and summary_line
mapping_counts, summary_line

```
Out[8]:
```
Summary of container-to-node-1 mapping from metric.csv:

- No metric rows were found where kpi_name contains "container.node-1".
- As a result: total unique cmdb_id values mapped to node-1 = 0.
- No earliest timestamp is available because there are no matching rows.

Suggested next steps (optional):
- Confirm the expected kpi_name pattern for node mapping (e.g., different capitalization or a different node identifier such as "container.node-5" used previously).
- If you expect pods on node-1, check other metric naming conventions or search for node-1 identifiers in other fields (e.g., cmdb_id naming or different metric prefixes).

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, metric_count]
Index: [], 'total_unique_cmdb_ids=0, earliest_metric_timestamp_for_node-1_mappings=')```
```

In [ ]:
# In[9]:

import re
import pandas as pd

# Reuse df_metric and prior aggregates if available
# Determine source list of cmdb_ids (limit to 50)
if 'breaches_by_cmdb_top20' in globals() and isinstance(breaches_by_cmdb_top20, pd.DataFrame) and not breaches_by_cmdb_top20.empty:
    source_cmdbs = breaches_by_cmdb_top20['cmdb_id'].astype(str).unique().tolist()[:50]
elif 'agg' in globals() and isinstance(agg, pd.DataFrame) and not agg.empty:
    source_cmdbs = agg.sort_values('num_breached_kpis', ascending=False)['cmdb_id'].astype(str).unique().tolist()[:50]
else:
    # fallback: top 50 cmdb_ids by frequency in df_metric
    source_cmdbs = df_metric['cmdb_id'].value_counts().head(50).index.astype(str).tolist()

# Pre-filter metric rows that contain any 'container.node-' token to reduce work
mask_node_tokens = df_metric['kpi_name'].str.contains(r'container\.node-\d+', case=False, na=False)
df_with_tokens = df_metric[mask_node_tokens].copy()

# Function to extract distinct node tokens from a series of kpi_name strings
token_re = re.compile(r'container\.node-\d+', re.IGNORECASE)

def extract_tokens(kpi_series):
    tokens = set()
    for s in kpi_series.dropna().astype(str):
        found = token_re.findall(s)
        for f in found:
            tokens.add(f.lower())
    return tokens

# Build mapping for selected cmdb_ids
rows = []
for cmdb in source_cmdbs:
    sub = df_with_tokens[df_with_tokens['cmdb_id'] == cmdb]
    tokens = extract_tokens(sub['kpi_name'])
    token_list = sorted(tokens)
    token_str = ",".join(token_list)
    rows.append({'cmdb_id': cmdb, 'node_tokens': token_str, 'count_node_tokens': len(token_list)})

mapping_df = pd.DataFrame(rows)

# Order rows by count_node_tokens desc then cmdb_id
mapping_df = mapping_df.sort_values(['count_node_tokens','cmdb_id'], ascending=[False, True]).reset_index(drop=True)

# Limit to top 50 rows (already limited by source_cmdbs but ensure)
mapping_df_top50 = mapping_df.head(50).copy()

# Summary: how many include 'container.node-1'
includes_node1 = mapping_df_top50['node_tokens'].str.contains(r'\bcontainer\.node-1\b', na=False).sum()
summary_line = f"cmdb_ids_with_container.node-1_in_tokens={int(includes_node1)} out of {mapping_df_top50.shape[0]} scanned (top 50)"

# Expose compact outputs
mapping_df_top50, summary_line

```
Out[9]:
```
Summary of pod → node mapping (scanned top cmdb_ids):

- Most pods are mapped to node-6: cartservice-1, cartservice-2, checkoutservice-0, checkoutservice-1, currencyservice-2, currencyservice2-0, frontend-0, frontend2-0, productcatalogservice-0/1/2 all show token "container.node-6".
- Several pods are mapped to node-5: checkoutservice-2, frontend-1, frontend-2 show "container.node-5".
- Some entries (e.g., adservice2 and the node_* cmdb_id rows) have no container.node-* tokens found.
- Importantly: none of the scanned cmdb_ids include "container.node-1".

Summary values:
- Scanned (top) cmdb_ids: 20 (limited selection).
- Number that include "container.node-1": 0 out of 20.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id       node_tokens  count_node_tokens
0             cartservice-1  container.node-6                  1
1             cartservice-2  container.node-6                  1
2         checkoutservice-0  container.node-6                  1
3         checkoutservice-1  container.node-6                  1
4         checkoutservice-2  container.node-5                  1
5         currencyservice-2  container.node-6                  1
6        currencyservice2-0  container.node-6                  1
7                frontend-0  container.node-6                  1
8                frontend-1  container.node-5                  1
9                frontend-2  container.node-5                  1
10              frontend2-0  container.node-6                  1
11  productcatalogservice-0  container.node-6                  1
12  productcatalogservice-1  container.node-6                  1
13  productcatalogservice-2  container.node-6                  1
14               adservice2                                    0
15                   node-1                                    0
16                   node-3                                    0
17                   node-4                                    0
18                   node-5                                    0
19                   node-6                                    0, 'cmdb_ids_with_container.node-1_in_tokens=0 out of 20 scanned (top 50)')```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Ensure df_metric is loaded and dt column (UTC+8) exists
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = pd.to_datetime(df_metric['dt'], utc=True).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# Nodes to analyze
nodes = [f"node-{i}" for i in range(1,7)]

# Keywords (case-sensitive as requested)
keywords = ['net','tcp','udp','retrans','packets','bytes']
pattern = '|'.join([kw for kw in keywords])

# Incident window (inclusive)
start = pd.Timestamp('2022-03-20 22:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 22:30:00', tz=tz)

# 1) Select relevant node metric rows (for threshold computation we need full series per (node,kpi))
mask_node = df_metric['cmdb_id'].isin(nodes)
mask_kpi  = df_metric['kpi_name'].str.contains(pattern, na=False)  # case-sensitive by default
df_nodes_all = df_metric[mask_node & mask_kpi].copy()

# If none, prepare empty outputs
if df_nodes_all.empty:
    node_summary = pd.DataFrame(columns=['node','num_breached_kpis','total_breach_points','overall_earliest_anom_ts','max_severity_overall','longest_consecutive_overall'])
    summary_line = "No network-related metrics found for node-1..node-6"
else:
    # 2) Compute global p95 and p5 per (node, kpi_name) using full series (do NOT filter by time)
    thr = df_nodes_all.groupby(['cmdb_id','kpi_name'])['value'].quantile([0.95,0.05]).unstack(level=-1).reset_index()
    # Rename columns (quantile indexes may be floats)
    col_map = {}
    for c in thr.columns:
        if isinstance(c, float):
            if np.isclose(c, 0.95):
                col_map[c] = 'p95'
            elif np.isclose(c, 0.05):
                col_map[c] = 'p5'
    thr = thr.rename(columns=col_map)
    # Ensure p95 and p5 present; if not, compute explicitly
    if 'p95' not in thr.columns or 'p5' not in thr.columns:
        thr_p95 = df_nodes_all.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
        thr_p5  = df_nodes_all.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index(name='p5')
        thr = thr_p95.merge(thr_p5, on=['cmdb_id','kpi_name'])
    thresholds_nodes = thr[['cmdb_id','kpi_name','p95','p5']].copy()

    # 3) Restrict points to the incident window for these node metrics
    df_nodes_win = df_nodes_all[(df_nodes_all['dt'] >= start) & (df_nodes_all['dt'] <= end)].copy()

    # Merge thresholds into windowed rows
    df_nodes_win = df_nodes_win.merge(thresholds_nodes, on=['cmdb_id','kpi_name'], how='left')
    df_nodes_win = df_nodes_win.dropna(subset=['p95','p5'])

    # Flag anomalies: high if >= p95; low if <= p5
    df_nodes_win['is_high'] = df_nodes_win['value'] >= df_nodes_win['p95']
    df_nodes_win['is_low']  = df_nodes_win['value'] <= df_nodes_win['p5']
    df_nodes_win['is_anom'] = df_nodes_win['is_high'] | df_nodes_win['is_low']

    # 4) Aggregate per (node, kpi_name)
    g = df_nodes_win.groupby(['cmdb_id','kpi_name'])
    pair_stats = g.agg(
        count_in_window = ('value','size'),
        count_high = ('is_high','sum'),
        count_low  = ('is_low','sum'),
        max_in_window = ('value','max'),
        min_in_window = ('value','min'),
        p95 = ('p95','first'),
        p5  = ('p5','first')
    ).reset_index()

    # earliest anomalous timestamp per pair (from anomalous rows)
    anom_rows = df_nodes_win[df_nodes_win['is_anom']]
    if not anom_rows.empty:
        earliest_pair = anom_rows.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index(name='earliest_anom_ts')
        pair_stats = pair_stats.merge(earliest_pair, on=['cmdb_id','kpi_name'], how='left')
    else:
        pair_stats['earliest_anom_ts'] = pd.NaT

    # Compute severity_high = max_in_window / p95 (if count_high>0)
    def compute_sev_high(r):
        if r['count_high'] <= 0:
            return np.nan
        p95 = r['p95']
        mx = r['max_in_window']
        if pd.isna(p95) or p95 == 0:
            return np.inf if (mx>0) else np.nan
        return mx / p95

    pair_stats['severity_high'] = pair_stats.apply(compute_sev_high, axis=1)

    # Compute longest_consecutive_anom_minutes per pair (consider anomalous if is_anom)
    # Build a helper mapping from (cmdb,kpi) -> longest run
    longest_map = {}
    if not anom_rows.empty:
        for (cmdb,kpi), grp in anom_rows.groupby(['cmdb_id','kpi_name']):
            s = grp.sort_values('dt')['dt']
            diffs = s.diff().dt.total_seconds().fillna(0)
            groups = (diffs != 60).cumsum()
            run_lengths = groups.groupby(groups).size()
            longest_map[(cmdb,kpi)] = int(run_lengths.max())
    pair_stats['longest_consecutive_anom_minutes'] = pair_stats.apply(lambda r: longest_map.get((r['cmdb_id'], r['kpi_name']), 0), axis=1)

    # breach points per pair = count_high + count_low
    pair_stats['breach_points'] = pair_stats['count_high'] + pair_stats['count_low']

    # Determine which pairs are "breached" (severity_high>1 OR severity_low>1). We computed severity_high; compute severity_low similarly
    def compute_sev_low(r):
        if r['count_low'] <= 0:
            return np.nan
        p5 = r['p5']
        mn = r['min_in_window']
        if pd.isna(mn) or mn == 0:
            return np.inf if (p5>0) else np.nan
        return p5 / mn

    pair_stats['severity_low'] = pair_stats.apply(compute_sev_low, axis=1)
    pair_stats['max_severity_pair'] = pair_stats[['severity_high','severity_low']].fillna(0).max(axis=1)

    # 5) Aggregate per node
    node_agg = pair_stats.groupby('cmdb_id').agg(
        num_breached_kpis = ('max_severity_pair', lambda s: int((s>1).sum())),
        total_breach_points = ('breach_points','sum'),
        overall_earliest_anom_ts = ('earliest_anom_ts','min'),
        max_severity_overall = ('max_severity_pair','max'),
        longest_consecutive_overall = ('longest_consecutive_anom_minutes','max')
    ).reset_index().rename(columns={'cmdb_id':'node'})

    # Format earliest timestamp strings
    node_agg['overall_earliest_anom_ts'] = node_agg['overall_earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Order rows as requested: by max_severity_overall desc then num_breached_kpis desc
    node_summary = node_agg.sort_values(['max_severity_overall','num_breached_kpis'], ascending=[False, False]).reset_index(drop=True)

    # Keep only requested columns and limit to 6 rows
    node_summary = node_summary.loc[:, ['node','num_breached_kpis','total_breach_points','overall_earliest_anom_ts','max_severity_overall','longest_consecutive_overall']].head(6).copy()

    # 6) One-line summary: identify node with largest max_severity_overall
    if not node_summary.empty:
        top_idx = node_summary['max_severity_overall'].idxmax()
        top_row = node_summary.loc[top_idx]
        summary_line = (f"top_node={top_row['node']}, max_severity_overall={top_row['max_severity_overall']}, "
                        f"earliest_anom_ts={top_row['overall_earliest_anom_ts']}, "
                        f"longest_consecutive_overall={int(top_row['longest_consecutive_overall'])}")
    else:
        summary_line = "No node network KPI anomalies found in the incident window."

# Return compact outputs
node_summary, summary_line

```
Out[10]:
```
Summary comparison of node-level network faults (window 2022-03-20 22:00–22:30 UTC+8):

- Overall result (per node):
  - node-1: num_breached_kpis=9, total_breach_points=502, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall=25.8, longest_consecutive_overall=31
  - node-5: num_breached_kpis=8, total_breach_points=506, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall≈9.684, longest_consecutive_overall=31
  - node-6: num_breached_kpis=9, total_breach_points=466, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall≈1.820, longest_consecutive_overall=31
  - node-3: num_breached_kpis=9, total_breach_points=479, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall≈1.456, longest_consecutive_overall=31
  - node-4: num_breached_kpis=8, total_breach_points=462, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall≈1.129, longest_consecutive_overall=31
  - node-2: num_breached_kpis=8, total_breach_points=470, earliest_anom_ts=2022-03-20 22:00:00, max_severity_overall≈1.111, longest_consecutive_overall=31

- Top node: node-1 — it has the largest max severity (25.8), earliest anomalous timestamp 2022-03-20 22:00:00, and the longest consecutive anomaly span recorded (31 minutes).

Interpretation and recommendations:
- node-1 exhibits the strongest network fault signal (very high severity — e.g., large TCP retransmission spike). node-5 also shows a strong fault signal (severity ≈ 9.68). Other nodes show smaller but notable anomalies.
- The earliest anomalies start at 22:00 on all affected nodes, and the longest consecutive anomalous durations reach the full inspected window (31 minutes) for several KPIs — this suggests sustained network degradation during the incident window.
- Immediate next steps: prioritize investigation on node-1 (then node-5) — check NIC/interface errors, driver and kernel logs, switch/router health, and which pods were placed on those nodes; correlate with traces/logs for impacted services to confirm user impact and root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(     node  num_breached_kpis  total_breach_points overall_earliest_anom_ts  max_severity_overall  longest_consecutive_overall
0  node-1                  9                  502  2022-03-20 22:00:00 CST             25.800000                           31
1  node-5                  8                  506  2022-03-20 22:00:00 CST              9.683784                           31
2  node-6                  9                  466  2022-03-20 22:00:00 CST              1.819675                           31
3  node-3                  9                  479  2022-03-20 22:00:00 CST              1.456023                           31
4  node-4                  8                  462  2022-03-20 22:00:00 CST              1.128958                           31
5  node-2                  8                  470  2022-03-20 22:00:00 CST              1.111111                           31, 'top_node=node-1, max_severity_overall=25.8, earliest_anom_ts=2022-03-20 22:00:00 CST, longest_consecutive_overall=31')```
```